<a href="https://colab.research.google.com/github/hmmnyamminji/python/blob/main/%EC%84%9C%EC%9A%B8%EC%8B%9C%EC%A7%80%ED%95%98%EC%B2%A0%EA%B5%90%ED%86%B5%EB%9F%89%EC%98%88%EC%B8%A1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### 환경 설정

In [ ]:
# 구글 드라이브 마운트
from google.colab import drive
drive.mount('/content/drive')

# 드라이브에 저장된 폰트 등록
import matplotlib as mpl
import matplotlib.pyplot as plt # 그래프를 그리는 pyplot 모듈
import matplotlib.font_manager as fm # 폰트 관리 모듈

# 드라이브 내 폰트 경로
font_path = '/content/drive/MyDrive/kwu/Bigdata/dataPreProcessing/NanumGothic.ttf'

fm.fontManager.addfont(font_path)
mpl.rc('font',family='NanumGothic') #matplotlib 기본 폰트로 설정
plt.rcParams['font.family'] = 'NanumGothic'
plt.rcParams['font.sans-serif'] = ['NanumGothic', 'sans-serif']
plt.rcParams['axes.unicode_minus'] = False # 마이너스(-) 기호가 깨지지 않도록 비활성화

print("현재 폰트: ", plt.rcParams['font.family'])

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
현재 폰트:  ['NanumGothic']


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import requests
import xml.etree.ElementTree as ET
import warnings # 코드 실행 중 발생하는 불피요한 경고 메시지를 숨김
warnings.filterwarnings('ignore')
import pprint

### API에서 직접 데이터 수집하는 방법

In [59]:
#XML방식
api_key = '504c6e584f67696d34376a72614e41'
start=1 #시작 행 번호
end=1000 #끝 행 번호

url = (f'http://openapi.seoul.go.kr:8088/{api_key}/xml'
      f'/CardSubwayTime/{start}/{end}/202601/경원선/광운대')
#url = (f'http://openapi.seoul.go.kr:8088/{api_key}/xml'
#      f'/TbUseMonthstatusView/{start}/{end}/')

response = requests.get(url, timeout=10) #HTTP GET 요청을 보내고 응답을 받음, 최대 10초 대기
response

#XML 바이트 -> 트리 구조 파싱
root = ET.fromstring(response.content)

#전체 건수 확인 (list_total_count 태그)
total_tag = root.find('.//list_total_count') #.// 하위 해당 태그를 모두 찾음
if total_tag is not None:
  print(f'전체 데이터 건수: {total_tag.text}')

rows = []
for row in root.findall('.//row'): #모든 row태그 탐색
  rows.append({child.tag:child.text for child in row}) #각 <row>태그 내의 자식 태그(컬럼명)와 텍스트(설명)

df=pd.DataFrame(rows)

전체 데이터 건수: 1


In [61]:
print(df.columns)


Index(['USE_MM', 'SBWY_ROUT_LN_NM', 'STTN', 'HR_4_GET_ON_NOPE',
       'HR_4_GET_OFF_NOPE', 'HR_5_GET_ON_NOPE', 'HR_5_GET_OFF_NOPE',
       'HR_6_GET_ON_NOPE', 'HR_6_GET_OFF_NOPE', 'HR_7_GET_ON_NOPE',
       'HR_7_GET_OFF_NOPE', 'HR_8_GET_ON_NOPE', 'HR_8_GET_OFF_NOPE',
       'HR_9_GET_ON_NOPE', 'HR_9_GET_OFF_NOPE', 'HR_10_GET_ON_NOPE',
       'HR_10_GET_OFF_NOPE', 'HR_11_GET_ON_NOPE', 'HR_11_GET_OFF_NOPE',
       'HR_12_GET_ON_NOPE', 'HR_12_GET_OFF_NOPE', 'HR_13_GET_ON_NOPE',
       'HR_13_GET_OFF_NOPE', 'HR_14_GET_ON_NOPE', 'HR_14_GET_OFF_NOPE',
       'HR_15_GET_ON_NOPE', 'HR_15_GET_OFF_NOPE', 'HR_16_GET_ON_NOPE',
       'HR_16_GET_OFF_NOPE', 'HR_17_GET_ON_NOPE', 'HR_17_GET_OFF_NOPE',
       'HR_18_GET_ON_NOPE', 'HR_18_GET_OFF_NOPE', 'HR_19_GET_ON_NOPE',
       'HR_19_GET_OFF_NOPE', 'HR_20_GET_ON_NOPE', 'HR_20_GET_OFF_NOPE',
       'HR_21_GET_ON_NOPE', 'HR_21_GET_OFF_NOPE', 'HR_22_GET_ON_NOPE',
       'HR_22_GET_OFF_NOPE', 'HR_23_GET_ON_NOPE', 'HR_23_GET_OFF_NOPE',
       'HR_0_GET_

In [60]:
df

,USE_MM,SBWY_ROUT_LN_NM,STTN,HR_4_GET_ON_NOPE,HR_4_GET_OFF_NOPE,HR_5_GET_ON_NOPE,HR_5_GET_OFF_NOPE,HR_6_GET_ON_NOPE,HR_6_GET_OFF_NOPE,HR_7_GET_ON_NOPE,...,HR_23_GET_OFF_NOPE,HR_0_GET_ON_NOPE,HR_0_GET_OFF_NOPE,HR_1_GET_ON_NOPE,HR_1_GET_OFF_NOPE,HR_2_GET_ON_NOPE,HR_2_GET_OFF_NOPE,HR_3_GET_ON_NOPE,HR_3_GET_OFF_NOPE,JOB_YMD
0,202601,경원선,광운대,2121,1,6310,1169,14307,3011,31022,...,9130,163,5117,0,2,0,0,0,0,20260203


In [ ]:
# JSON 방식
url = (f'http://openapi.seoul.go.kr:8088/{api_key}/json/'f'TbUseYearstatusView/{start}/{end}/')

data = requests.get(url, timeout=10).json() #JSON 방식으로 자동 파싱
pprint.pprint(data)

svc = data.get('TbUseMonthstatusView', {}) # 키의 값을 가져오기
print(f'전체 데이터 건수: {svc.get("list_total_count","알 수 없음")}')

df_j = pd.DataFrame(svc.get('row',[]))

{'TbUseYearstatusView': {'RESULT': {'CODE': 'INFO-000',
                                    'MESSAGE': '정상 처리되었습니다'},
                         'list_total_count': 197,
                         'row': [{'DSTRCT_TYPE': 'PLT-004',
                                  'DT': '2026',
                                  'PKLT_NM': '이촌1주차장',
                                  'PRK_CNTOM': 15047.0,
                                  'UTZTN_HR': 1369814.0},
                                 {'DSTRCT_TYPE': 'PLT-006',
                                  'DT': '2026',
                                  'PKLT_NM': '광나루1,2주차장',
                                  'PRK_CNTOM': 36526.0,
                                  'UTZTN_HR': 3331937.0},
                                 {'DSTRCT_TYPE': 'PLT-007',
                                  'DT': '2026',
                                  'PKLT_NM': '양화1주차장',
                                  'PRK_CNTOM': 16614.0,
                                  'UTZTN_HR': 1843016.0}

In [ ]:
df_j

""


In [ ]:
# csv 파일로 저장
BASE = '/content/drive/MyDrive/kwu/Bigdata/eda/서울시 지하철 호선별 역별 시간대별 승하차 인원 정보.csv'
PATH_DATA = BASE + 'TbUseMonthstatusView.csv'

df_j.to_csv(PATH_DATA, index=False, encoding='utf-8-sig')